## Paso 1 — Cargamos el dataset ya tipado

Leemos directamente el Parquet generado en `00_setup_y_carga.ipynb`. Este
notebook cubre la Fase 3 (unicidad y granularidad, §4 del plan) y la Fase 4
(calidad del dato, §5 del plan).


In [564]:
import pandas as pd
from pathlib import Path

INTERIM_DIR = Path("../datos/interim")
df = pd.read_parquet(INTERIM_DIR / "viajes_btura_tipado.parquet")

print(df.shape)


(41237, 17)


## Paso 2 — Duplicados de fila completa

¿Hay filas idénticas en las 17 columnas a la vez? Esto es distinto al
problema de `fo` repetido (ahí algo sí cambia entre las filas) — aquí
buscamos copias exactas, que serían un defecto de extracción más grave.


In [565]:
duplicados_completos = df.duplicated(keep=False)
print(f"Filas duplicadas por completo: {duplicados_completos.sum()}")

Filas duplicadas por completo: 0


## Paso 3 — Duplicados de la clave `fo` y qué cambia entre ellos

Confirmamos el conteo de `fo` repetidos, y verificamos de forma sistemática
—sobre los 130 grupos completos, no solo un par de ejemplos— qué columna(s)
cambian dentro de cada grupo repetido.

In [566]:
conteo_fo = df["fo"].value_counts()
fo_repetidos = conteo_fo[conteo_fo > 1].index

print(f"Valores de 'fo' únicos:         {df['fo'].nunique()}")
print(f"Valores de 'fo' repetidos:      {len(fo_repetidos)}")
print(f"Filas involucradas en repetidos: {df['fo'].isin(fo_repetidos).sum()}")

Valores de 'fo' únicos:         41107
Valores de 'fo' repetidos:      128
Filas involucradas en repetidos: 258


In [567]:
columnas_a_revisar = [c for c in df.columns if c != "fo"]

variacion_por_columna = {col: 0 for col in columnas_a_revisar}
for fo_val, grupo in df[df["fo"].isin(fo_repetidos)].groupby("fo"):
    for col in columnas_a_revisar:
        if grupo[col].nunique(dropna=False) > 1:
            variacion_por_columna[col] += 1

variacion_df = pd.Series(variacion_por_columna, name="grupos_con_variacion").sort_values(ascending=False)
variacion_df

pesoNeto                 105
fleteProduccionFo        102
tarifaVentaAlCliente      99
bpCliente                 96
claseEquipoFop            48
destino                   34
grupoDeEquipoFwop         33
posicionPrincipalFwop     32
origen                    25
distanciaTotal            17
fechaCreacionFo            0
idClaseDocumentoFwo        0
idMedioTransporteFo        0
vehiculo                   0
tipo                       0
propiedad                  0
Name: grupos_con_variacion, dtype: int64

In [568]:
grupos_bp_distinto = [
    fo_val for fo_val, grupo in df[df["fo"].isin(fo_repetidos)].groupby("fo")
    if grupo["bpCliente"].nunique(dropna=False) > 1
]

print(f"Grupos donde bpCliente varía: {len(grupos_bp_distinto)}")
df[df["fo"].isin(grupos_bp_distinto[:8])].sort_values("fo")


Grupos donde bpCliente varía: 96


,idClaseDocumentoFwo,fo,fechaCreacionFo,bpCliente,origen,destino,distanciaTotal,pesoNeto,posicionPrincipalFwop,grupoDeEquipoFwop,vehiculo,idMedioTransporteFo,claseEquipoFop,propiedad,tipo,tarifaVentaAlCliente,fleteProduccionFo
38265,OPIM,301204560,2025-04-01,6,"BOGOTA, DISTRITO CAPITAL",BUENAVENTURA,277.5,2300.0,TUR,CN,JTZ423,TC-2S2,CNT20.ST,Tercero,TC.2S2,600000.0,600000.0
39688,OPIM,301204560,2025-04-01,761,CAJICA,BUENAVENTURA,277.5,2300.0,TUR,CN,JTZ423,TC-2S2,CNT20.ST,Tercero,TC.2S2,600000.0,600000.0
38273,OPIM,301205267,2025-04-02,6,TOCANCIPA,BUENAVENTURA,260.5,2300.0,TUR,CN,SYL355,TC-3S3,CNT20.ST,Tercero,TC.3S3,750000.0,750000.0
34610,OPIM,301205267,2025-04-02,1910,FUNZA,BUENAVENTURA,260.5,2300.0,TUR,CN,SYL355,TC-3S3,CNT20.ST,Tercero,TC.3S3,750000.0,750000.0
38276,OPIM,301205855,2025-04-03,6,"BOGOTA, DISTRITO CAPITAL",BUENAVENTURA,250.5,2300.0,TUR,CN,SZA438,TC-3S3,CNT20.ST,Tercero,TC.3S3,600000.0,600000.0
31043,OPIM,301205855,2025-04-03,104369,FUNZA,BUENAVENTURA,250.5,2300.0,TUR,CN,SZA438,TC-3S3,CNT20.ST,Tercero,TC.3S3,0.0,600000.0
23314,OPIM,301209857,2025-04-11,584,BUENAVENTURA,"BOGOTA, DISTRITO CAPITAL",248.5,27040.0,TUR,CN,TGK823,TC-3S3,CNT20.ST,Tercero,TC.3S3,5950000.0,5079151.0
6767,OPIM,301209857,2025-04-11,15964,BUENAVENTURA,"BOGOTA, DISTRITO CAPITAL",248.5,2000.0,PKG,NaN,TGK823,TC-3S3,NaN,Tercero,TC.3S3,620849.0,620849.0
34083,OPIM,301211301,2025-04-14,16955,COTA,BUENAVENTURA,268.5,2300.0,TUR,CN,SVB448,TC-2S2,CNT20.ST,Tercero,TC.2S2,600000.0,600000.0
30970,OPIM,301211301,2025-04-14,1017,FUNZA,BUENAVENTURA,268.5,2300.0,TUR,CN,SVB448,TC-2S2,CNT20.ST,Tercero,TC.2S2,820000.0,600000.0


In [569]:
columnas_estables_esperadas = ["fechaCreacionFo", "vehiculo", "distanciaTotal", "pesoNeto"]
columnas_variables_esperadas = ["bpCliente", "origen"]

patron_encontrado = 0
for fo_val, grupo in df[df["fo"].isin(fo_repetidos)].groupby("fo"):
    estables_ok = all(grupo[c].nunique(dropna=False) == 1 for c in columnas_estables_esperadas)
    variables_ok = any(grupo[c].nunique(dropna=False) > 1 for c in columnas_variables_esperadas)
    if estables_ok and variables_ok:
        patron_encontrado += 1

print(f"Grupos con 'mismo vehículo/fecha/distancia/peso pero cliente u origen distinto': {patron_encontrado} de {len(fo_repetidos)}")


Grupos con 'mismo vehículo/fecha/distancia/peso pero cliente u origen distinto': 23 de 128


In [570]:
patron_amplio = 0
for fo_val, grupo in df[df["fo"].isin(fo_repetidos)].groupby("fo"):
    mismo_vehiculo_fecha = (grupo["vehiculo"].nunique() == 1) and (grupo["fechaCreacionFo"].nunique() == 1)
    cliente_o_peso_distinto = (
        grupo["bpCliente"].nunique(dropna=False) > 1
        or grupo["pesoNeto"].nunique(dropna=False) > 1
    )
    if mismo_vehiculo_fecha and cliente_o_peso_distinto:
        patron_amplio += 1

print(f"Grupos con mismo vehículo/fecha, pero cliente o peso distinto: {patron_amplio} de {len(fo_repetidos)}")


Grupos con mismo vehículo/fecha, pero cliente o peso distinto: 128 de 128


## Síntesis del Paso 3 — Unicidad de `fo`

**Hallazgo que corrige al plan (§4.2):** el plan afirmaba, con base en 2
ejemplos, que los 130 registros de `fo` repetido difieren únicamente en
`Origen`. La verificación sistemática sobre los 128 grupos completos lo
descarta: `origen` cambia en solo 25 de 128 (20%), mientras que `pesoNeto`
(105), `fleteProduccionFo` (102), `tarifaVentaAlCliente` (99) y `bpCliente`
(96) cambian en la mayoría de los casos.

**Patrón universal confirmado (128 de 128, 100%):** todo grupo de `fo`
repetido comparte el mismo `vehiculo` y la misma `fechaCreacionFo`, pero
difiere en `bpCliente` y/o `pesoNeto` (y casi siempre en las columnas de
plata).

**Hipótesis más consistente:** `fo` identifica el **viaje del vehículo**
(así lo define el diccionario interno — "documento de subcontratación con
el dueño del vehículo"), no la orden de un cliente individual. Un mismo
viaje puede consolidar carga de varios clientes, cada uno con su propio
peso y tarifa, bajo un solo número de `fo`.

**Sin resolver:** al menos un caso (`fo` 301205267) muestra cliente
distinto con tarifa idéntica entre las dos filas — no se puede descartar
que sea una corrección de cliente mal aplicada en vez de carga consolidada
genuina.

**Pregunta a escalar al negocio (reemplaza la pregunta genérica del plan):**
"¿Un mismo número de `fo` puede representar el viaje de un vehículo que
transporta carga de varios clientes distintos (carga consolidada)? ¿O cada
`fo` debería corresponder a un único cliente, y estos 128 casos son un
defecto de la extracción/sistema?"

**Implicación práctica (pendiente de confirmar):** si la hipótesis de carga
consolidada es correcta, sumar `tarifaVentaAlCliente`/`fleteProduccionFo`
por fila (no por `fo` único) sería lo correcto para métricas de ingreso —
al revés de lo que advertía el plan original ("no sumar, se duplicaría el
ingreso"). No usar para reportes hasta confirmar con negocio.


## Paso 4 — Clave compuesta única

`fo` solo no es único. Probamos combinaciones de columnas hasta encontrar
una que sí identifique cada fila sin ambigüedad.


In [571]:
candidatos = [
    ["fo"],
    ["fo", "bpCliente"],
    ["fo", "bpCliente", "pesoNeto"],
    ["fo", "vehiculo"],
]

for combinacion in candidatos:
    n_combinaciones_unicas = df[combinacion].drop_duplicates().shape[0]
    es_unica = n_combinaciones_unicas == len(df)
    print(f"{combinacion}: {n_combinaciones_unicas} combinaciones únicas de {len(df)} filas → {'ÚNICA' if es_unica else 'no única'}")


['fo']: 41107 combinaciones únicas de 41237 filas → no única
['fo', 'bpCliente']: 41204 combinaciones únicas de 41237 filas → no única
['fo', 'bpCliente', 'pesoNeto']: 41236 combinaciones únicas de 41237 filas → no única
['fo', 'vehiculo']: 41107 combinaciones únicas de 41237 filas → no única


In [572]:
clave = ["fo", "bpCliente", "pesoNeto"]
duplicados_clave = df[df.duplicated(subset=clave, keep=False)].sort_values(clave)
duplicados_clave


,idClaseDocumentoFwo,fo,fechaCreacionFo,bpCliente,origen,destino,distanciaTotal,pesoNeto,posicionPrincipalFwop,grupoDeEquipoFwop,vehiculo,idMedioTransporteFo,claseEquipoFop,propiedad,tipo,tarifaVentaAlCliente,fleteProduccionFo
1841,OPIM,301351058,2025-12-22,113631,BUENAVENTURA,LA ESTRELLA,184.0,5000.0,PKG,NaN,TRK102,FP-2S2,NaN,Propio,FP.2S2,2400000.0,2400000.0
1842,OPIM,301351058,2025-12-22,113631,BUENAVENTURA,MEDELLIN,184.0,5000.0,PKG,NaN,TRK102,FP-2S2,NaN,Propio,FP.2S2,2400000.0,2400000.0


In [573]:
clave_final = ["fo", "bpCliente", "pesoNeto", "destino"]
n_unicas = df[clave_final].drop_duplicates().shape[0]
print(f"{clave_final}: {n_unicas} de {len(df)} → {'ÚNICA' if n_unicas == len(df) else 'no única'}")


['fo', 'bpCliente', 'pesoNeto', 'destino']: 41237 de 41237 → ÚNICA


## Paso 5 — Vehículo en la misma fecha con rutas distintas

Sin horas reales, no podemos confirmar solapamiento físico (limitación ya
documentada en L-01 del plan). Como proxy, revisamos si un mismo vehículo,
en la misma fecha, aparece con más de una distancia distinta — señal de
dos viajes genuinamente diferentes, no de una carga consolidada (que
comparte la misma distancia).


In [574]:
resultados_vehiculo = []
for (vehiculo_val, fecha_val), grupo in df.groupby(["vehiculo", "fechaCreacionFo"]):
    distancias_distintas = grupo["distanciaTotal"].nunique()
    if distancias_distintas > 1:
        resultados_vehiculo.append({
            "vehiculo": vehiculo_val,
            "fecha": fecha_val,
            "distancias_distintas": distancias_distintas,
            "filas": len(grupo),
        })

sospechosos = pd.DataFrame(resultados_vehiculo)
print(f"Combinaciones vehículo+fecha con más de una distancia: {len(sospechosos)}")
sospechosos.sort_values("distancias_distintas", ascending=False).head(10)


Combinaciones vehículo+fecha con más de una distancia: 3079


,vehiculo,fecha,distancias_distintas,filas
708,SKE945,2025-11-25,4,4
1928,TMP377,2025-04-25,4,4
3038,XJA785,2026-04-01,4,4
2954,WRD415,2026-02-26,4,4
185,JUN988,2025-08-27,3,3
291,KOL591,2026-02-25,3,3
886,SNO363,2025-12-12,3,3
366,LPY725,2025-06-05,3,3
383,LPY725,2025-11-11,3,3
937,SNU273,2025-05-13,3,4


In [575]:
total_combinaciones = df.groupby(["vehiculo", "fechaCreacionFo"]).ngroups
print(f"Total combinaciones vehículo+fecha: {total_combinaciones}")
print(f"Con más de una distancia: {len(sospechosos)} ({len(sospechosos)/total_combinaciones*100:.1f}%)")

# miremos el primer ejemplo con detalle
ejemplo = df[(df["vehiculo"] == "SKE945") & (df["fechaCreacionFo"] == "2025-11-25")]
ejemplo[["fo", "idClaseDocumentoFwo", "origen", "destino", "distanciaTotal", "pesoNeto"]]


Total combinaciones vehículo+fecha: 30883
Con más de una distancia: 3079 (10.0%)


,fo,idClaseDocumentoFwo,origen,destino,distanciaTotal,pesoNeto
759,301334929,OPEX,BUENAVENTURA,YUMBO,156.0,4000.0
25665,301335565,OPIM,BUENAVENTURA,VILLA RICA,183.0,21000.0
31192,301334928,OPEX,YUMBO,BUENAVENTURA,147.0,28000.0
39383,301335566,OPIM,VILLA RICA,BUENAVENTURA,207.0,4000.0


## Síntesis del bloque — Unicidad y granularidad (§4 del plan)

**Duplicados de fila completa:** 0 (Paso 2). No hay copias exactas.

**`fo` no es clave única:** 128 valores repetidos de 41.107 (Paso 3).
Verificado sistemáticamente sobre los 128 grupos completos —no sobre una
muestra— el patrón es 100% consistente: mismo `vehiculo` y misma
`fechaCreacionFo`, pero difiere `bpCliente` y/o `pesoNeto`. Esto corrige al
plan original (§4.2): no es un error de duplicación ni "multi-recogida"
simple, es **carga consolidada de varios clientes en un mismo viaje de
vehículo**. Confirmado con el sistema fuente (tabla `FactFWOFUFO` en
DWCubos): `Distancia Total` es un atributo del viaje (`FO`), el origen es
un atributo de la posición del cliente (`FWOP`) — dos niveles jerárquicos
distintos que el extracto plano combina en una sola fila.

**Clave compuesta práctica identificada (Paso 4):**
`fo + bpCliente + pesoNeto + destino` identifica las 41.237 filas sin
colisión. `vehiculo` no aporta poder discriminador (nunca varía dentro de
un grupo de `fo` repetido).

**Solapamiento de vehículo en el tiempo (Paso 5):** no concluyente. El
único proxy disponible (`fechaCreacionFo`) es administrativo, no la fecha
de ejecución real — el ejemplo revisado (vehículo `SKE945`) mostró dos
viajes de ida y vuelta perfectamente normales, no una contradicción física.
Este chequeo no aporta un hallazgo nuevo; **reafirma la limitación L-01**
(sin fechas/horas de ejecución, no se puede analizar la operación en el
tiempo).

**Qué es "una fila" en este dataset:** ni un viaje completo (`fo`) ni un
cliente (`bpCliente`) — es una **línea de operación/facturación dentro de
un viaje**. Este matiz debe quedar explícito en el informe final (§12) y en
cualquier conteo de "número de viajes" (usar `fo.nunique()`, no
`len(df)`).

**Pendiente (no bloqueante):** el punto de §4.1 sobre "duplicados lógicos"
(mismo cliente/fecha/origen-destino/peso con `fo` distinto —el problema
inverso: fragmentación en vez de consolidación) no se investigó a fondo,
dado que la pregunta principal de granularidad ya quedó resuelta con alta
confianza. Se puede retomar si aparece evidencia de que hace falta.


## Paso 6 — Completitud: ¿son estructurales los nulos de equipo?

El diccionario dice que `grupoDeEquipoFwop` solo aplica a la posición TUR,
y `claseEquipoFop` solo a los grupos CN/CAM. Si es así, los nulos deberían
concentrarse exactamente en PKG y PRD, no estar dispersos. Lo verificamos
cruzando nulos contra `posicionPrincipalFwop`, en vez de solo mirar el %
global de nulos.


In [576]:
resumen_nulos = df.groupby("posicionPrincipalFwop").agg(
    total=("fo", "size"),
    grupoEquipo_nulo=("grupoDeEquipoFwop", lambda s: s.isna().sum()),
    claseEquipo_nulo=("claseEquipoFop", lambda s: s.isna().sum()),
)
resumen_nulos


,total,grupoEquipo_nulo,claseEquipo_nulo
posicionPrincipalFwop,,,
PKG,1299,1299,1299
PRD,6197,6197,6197
TUR,33741,0,3


In [577]:
defectos_reales = df[(df["posicionPrincipalFwop"] == "TUR") & (df["claseEquipoFop"].isna())]
print(f"Defectos reales (TUR con claseEquipoFop nula): {len(defectos_reales)}")
defectos_reales[["fo", "vehiculo", "idClaseDocumentoFwo", "grupoDeEquipoFwop", "idMedioTransporteFo", "claseEquipoFop"]]


Defectos reales (TUR con claseEquipoFop nula): 3


,fo,vehiculo,idClaseDocumentoFwo,grupoDeEquipoFwop,idMedioTransporteFo,claseEquipoFop
4398,301261226,YAP042,OHUB,CN,TC-3S3,NaN
4561,301295466,VBI966,OHUB,CN,TC-3S3,NaN
30020,301214738,VBI966,OHUB,CN,TC-3S3,NaN


## Síntesis del Paso 6 — Completitud

Confirmado con código: los nulos de `grupoDeEquipoFwop` (18,18%) y
`claseEquipoFop` (18,19%) son estructurales, exactamente como predice el
diccionario — 100% de PKG (1.299) y PRD (6.197) tienen ambas columnas
nulas por diseño, no por defecto. El defecto real son **3 registros TUR**
con `claseEquipoFop` nula (`fo` 301261226, 301295466, 301214738), todos
`OHUB` + `CN` + `TC-3S3`. Vehículo `VBI966` repite el defecto en 2 de los
3 casos. Severidad: baja (0,007% del set) — se listan para escalar, no
bloquean el análisis.


## Paso 7 — Nulos disfrazados

Un valor puede estar "vacío" sin ser `NaN` técnicamente — texto como "NA",
"-", "SIN DATO", o una fecha centinela como 1900-01-01. Revisamos las
columnas de texto contra una lista de placeholders comunes, y las fechas
contra años fuera de rango razonable.


In [578]:
placeholders_sospechosos = {"", "NA", "N/A", "-", "--", "SIN DATO", "NULL", "NONE", "S/D"}

columnas_texto = [
    "idClaseDocumentoFwo", "origen", "destino", "posicionPrincipalFwop",
    "grupoDeEquipoFwop", "vehiculo", "idMedioTransporteFo",
    "claseEquipoFop", "propiedad", "tipo",
]

encontrados = False
for col in columnas_texto:
    valores_unicos = df[col].dropna().unique()
    sospechosos = [v for v in valores_unicos if str(v).strip().upper() in placeholders_sospechosos]
    if sospechosos:
        print(f"{col}: {sospechosos}")
        encontrados = True

if not encontrados:
    print("Ningún placeholder sospechoso encontrado en las columnas de texto.")

fechas_sospechosas = df[df["fechaCreacionFo"].dt.year < 2000]
print(f"Fechas con año < 2000 (posible fecha centinela): {len(fechas_sospechosas)}")


Ningún placeholder sospechoso encontrado en las columnas de texto.
Fechas con año < 2000 (posible fecha centinela): 0


## Paso 8 — Validez: negativos, fechas fuera de ventana, formato de placas

Verificamos rangos físicos básicos: valores negativos en las numéricas,
fechas fuera de la ventana ya documentada (2025-04-01 a 2026-04-07), y si
las placas de vehículo siguen un patrón consistente. La revisión de
domingos/festivos con volumen anómalo se deja para el notebook temporal
(`05`), donde ya está planificada — evitamos repetir el mismo análisis dos
veces.


In [579]:
import re

# Negativos en las 4 numéricas
columnas_numericas = ["distanciaTotal", "pesoNeto", "tarifaVentaAlCliente", "fleteProduccionFo"]
for col in columnas_numericas:
    negativos = (df[col] < 0).sum()
    print(f"{col}: {negativos} valores negativos")

# Fechas fuera de la ventana ya documentada
ventana_inicio = pd.Timestamp("2025-04-01")
ventana_fin = pd.Timestamp("2026-04-07")
fuera_de_ventana = df[(df["fechaCreacionFo"] < ventana_inicio) | (df["fechaCreacionFo"] > ventana_fin)]
print(f"\nFechas fuera de la ventana declarada: {len(fuera_de_ventana)}")

# Formato de placas: patrón estándar colombiano (3 letras + 3 números)
patron_placa = re.compile(r"^[A-Z]{3}\d{3}$")
placas_no_estandar = df.loc[~df["vehiculo"].str.match(patron_placa, na=False), "vehiculo"].unique()
print(f"\nPlacas que no siguen el patrón 3 letras + 3 números: {len(placas_no_estandar)}")
print(placas_no_estandar[:20])


distanciaTotal: 0 valores negativos
pesoNeto: 0 valores negativos
tarifaVentaAlCliente: 0 valores negativos
fleteProduccionFo: 1 valores negativos

Fechas fuera de la ventana declarada: 0

Placas que no siguen el patrón 3 letras + 3 números: 3
<ArrowStringArray>
['A36AR1U', 'A99AG8R', 'GENERICO_3S3']
Length: 3, dtype: str


In [580]:
df[df["vehiculo"].isin(["A36AR1U", "A99AG8R", "GENERICO_3S3"])]

,idClaseDocumentoFwo,fo,fechaCreacionFo,bpCliente,origen,destino,distanciaTotal,pesoNeto,posicionPrincipalFwop,grupoDeEquipoFwop,vehiculo,idMedioTransporteFo,claseEquipoFop,propiedad,tipo,tarifaVentaAlCliente,fleteProduccionFo
13071,OPTI,301305965,2025-10-04,34052,BUENAVENTURA,VALENCIA(VE),615.0,25000.0,TUR,CN,A36AR1U,TC-3S3,CNT40.HC,Tercero,TC.3S3,12000000.0,5600000.0
13073,OPTI,301311872,2025-10-16,34052,BUENAVENTURA,CARACAS(VE),774.0,25000.0,TUR,CN,A99AG8R,TC-3S2,CNT40.HC,Tercero,TC.3S2,19048010.0,6847040.0
17046,OPIM,301395804,2026-03-17,56788,BUENAVENTURA,YUMBO,156.0,32000.0,PRD,NaN,GENERICO_3S3,TC-3S3,NaN,Tercero,TC.3S3,2754000.0,2400000.0


In [581]:
# ¿Es "GENERICO" un patrón que se repite, ligado a posiciones no-TUR?
genericos = df[df["vehiculo"].str.contains("GENERICO", na=False)]
print(f"Filas con 'GENERICO' en vehiculo: {len(genericos)}")
print(genericos["posicionPrincipalFwop"].value_counts())

# ¿Todos los OPTI (internacional) tienen placas no-colombianas?
opti = df[df["idClaseDocumentoFwo"] == "OPTI"]
print(f"\nTotal registros OPTI: {len(opti)}")
opti[["fo", "destino", "vehiculo"]]


Filas con 'GENERICO' en vehiculo: 1
posicionPrincipalFwop
PRD    1
Name: count, dtype: int64

Total registros OPTI: 22


,fo,destino,vehiculo
1901,301332981,IPIALES,LQO396
3502,301337543,TULCAN(EC),WMR619
3659,301316006,GUARERO(VE),KRM182
3660,301316031,GUARERO(VE),LQO460
3662,301321231,GUARERO(VE),SNR046
3663,301321244,GUARERO(VE),SEY436
13070,301297678,GUARERO(VE),SKF814
13071,301305965,VALENCIA(VE),A36AR1U
13072,301306449,GUARERO(VE),SOT089
13073,301311872,CARACAS(VE),A99AG8R


## Síntesis del Paso 8 — Validez

**Negativos:** 1 en `fleteProduccionFo` (el ya conocido -49.978, Anexo A).
Resto en 0.

**Fechas:** 0 fuera de la ventana declarada (2025-04-01 a 2026-04-07).

**Placas no estándar (3):**
- `A36AR1U` (destino Valencia, VE) y `A99AG8R` (destino Caracas, VE):
  **no son defecto** — son placas venezolanas legítimas. Confirmado
  geográficamente: de los 22 registros `OPTI` (internacional), los 20 con
  destino fronterizo (Tulcán-EC, Guarero-VE) usan placa colombiana normal;
  solo los 2 con destino profundo en Venezuela usan placa extranjera,
  coherente con un cambio de vehículo en frontera.
- `GENERICO_3S3` (fo 301395804, posición PRD): **sí es un defecto real**,
  aislado (1 de 41.237) — vehículo no asignado/desconocido, no un patrón
  estructural (otras filas PRD sí tienen placa real).


## Paso 9 — Exactitud: reglas de negocio (§5.5, Anexo C)

Reproducimos con código las 7 reglas de negocio del plan, usando la misma
numeración del Anexo C (R01-R07), para confirmar (o corregir) los números
citados como baseline.


In [582]:
reglas = {
    "R01_distancia_no_nula":       df["distanciaTotal"] > 0,
    "R02_peso_no_nulo":            df["pesoNeto"] > 0,
    "R03_tarifa_no_nula":          df["tarifaVentaAlCliente"] > 0,
    "R04_flete_no_nulo":           df["fleteProduccionFo"] > 0,
    "R05_flete_no_negativo":       df["fleteProduccionFo"] >= 0,
    "R06_margen_no_negativo":      df["tarifaVentaAlCliente"] >= df["fleteProduccionFo"],
    "R07_origen_distinto_destino": df["origen"] != df["destino"],
}

resumen_reglas = pd.DataFrame({
    "regla": list(reglas.keys()),
    "incumplen": [(~s).sum() for s in reglas.values()],
    "pct_incumplen": [round((~s).sum() / len(df) * 100, 2) for s in reglas.values()],
})
resumen_reglas


,regla,incumplen,pct_incumplen
0,R01_distancia_no_nula,17,0.04
1,R02_peso_no_nulo,82,0.20
2,R03_tarifa_no_nula,874,2.12
3,R04_flete_no_nulo,184,0.45
4,R05_flete_no_negativo,1,0.00
5,R06_margen_no_negativo,3839,9.31
6,R07_origen_distinto_destino,8966,21.74


## Síntesis del Paso 9 — Exactitud (reglas de negocio)

Las 7 reglas del Anexo C reproducidas con código coinciden exactamente con
el baseline del plan (Anexo A): R01=17, R02=82, R03=874, R04=184 (183 en
cero + 1 negativo), R05=1 (el negativo aislado), R06=3.839 (9,31% con
margen negativo), R07=8.966 (21,74% con Origen=Destino). Ningún número
requirió corrección.

Pendiente de este bloque: R06 (margen no negativo) y R07 (Origen=Destino)
tienen explicaciones ya conocidas del plan (carga consolidada, OHUB/OH3P) o
requieren el cruce contra el catálogo CLASE FWO — se detalla en el
siguiente paso.



## Paso 10 — Cruce idClaseDocumentoFwo × Origen = Destino

El 21,74% de Origen=Destino no es un solo fenómeno. Lo desagregamos por
clase de documento para separar lo legítimo (descontenerización) de lo
que sí contradice la definición de su clase.


In [583]:
df["origen_igual_destino"] = df["origen"] == df["destino"]

cruce = df.groupby("idClaseDocumentoFwo").agg(
    total=("fo", "size"),
    origen_eq_destino=("origen_igual_destino", "sum"),
)
cruce["pct_dentro_de_clase"] = (cruce["origen_eq_destino"] / cruce["total"] * 100).round(1)
cruce["pct_del_total_oed"] = (cruce["origen_eq_destino"] / cruce["origen_igual_destino"].sum() * 100).round(1) if False else (cruce["origen_eq_destino"] / df["origen_igual_destino"].sum() * 100).round(1)
cruce.sort_values("origen_eq_destino", ascending=False)


,total,origen_eq_destino,pct_dentro_de_clase,pct_del_total_oed
idClaseDocumentoFwo,,,,
OHUB,6092,6082,99.8,67.8
OH3P,1871,1871,100.0,20.9
OPIM,27813,832,3.0,9.3
OPTN,1497,146,9.8,1.6
OPEX,3877,34,0.9,0.4
OPDS,65,1,1.5,0.0
OPTI,22,0,0.0,0.0


In [584]:
defectos_oed = df[df["idClaseDocumentoFwo"].isin(["OPIM", "OPTN"]) & df["origen_igual_destino"]]
print(f"Defecto real (OPIM+OPTN con Origen=Destino): {len(defectos_oed)}")

Defecto real (OPIM+OPTN con Origen=Destino): 978


In [585]:
# 1) Cruce general: distancia=0 vs origen=destino
print(pd.crosstab(df["distanciaTotal"] == 0, df["origen_igual_destino"], 
                   rownames=["distancia_0"], colnames=["origen_eq_destino"]))

# 2) Dentro de los 978 defectos, ¿qué distancias tienen?
print("\nDistribución de distanciaTotal en los 978 defectos OPIM+OPTN:")
print(defectos_oed["distanciaTotal"].describe())


origen_eq_destino  False  True 
distancia_0                    
False              32257   8963
True                  14      3

Distribución de distanciaTotal en los 978 defectos OPIM+OPTN:
count    978.000000
mean      11.779652
std       28.866717
min       10.000000
25%       10.000000
50%       10.000000
75%       10.000000
max      521.000000
Name: distanciaTotal, dtype: float64


In [586]:
# Los 14 casos más contradictorios: distancia=0 con origen distinto de destino
casos_raros = df[(df["distanciaTotal"] == 0) & (~df["origen_igual_destino"])]
print(casos_raros[["fo", "idClaseDocumentoFwo", "origen", "destino", "pesoNeto"]])

# ¿Cuántos de los 978 tienen exactamente distancia=10 (firma de descontenerización)?
print(f"\n{(defectos_oed['distanciaTotal'] == 10).sum()} de 978 con distancia=10")
print(defectos_oed["distanciaTotal"].value_counts().head(10))

              fo idClaseDocumentoFwo        origen       destino  pesoNeto
945    301379586                OPIM  BUENAVENTURA         YUMBO   26140.0
2673   301405796                OPIM  BUENAVENTURA         FUNZA   22500.0
6447   301273170                OPIM  BUENAVENTURA      MOSQUERA    4000.0
6583   301325265                OPIM  BUENAVENTURA       SOLEDAD    4096.0
6657   301353744                OPIM  BUENAVENTURA   BUCARAMANGA    4688.0
9017   301405671                OPIM  BUENAVENTURA     CHINCHINA   33000.0
10141  301364687                OPIM  BUENAVENTURA       SOLEDAD    4830.0
17015  301393848                OPIM  BUENAVENTURA         YUMBO   34000.0
30588  301394128                OPIM  BUENAVENTURA         TULUA   34000.0
32309  301365489                OPEX      CORRALES  BUENAVENTURA   34210.0
33451  301405272                OPEX     MANIZALES  BUENAVENTURA   12476.0
33808  301313723                OPIM         YUMBO  BUENAVENTURA    4000.0
34748  301261603         

In [592]:
print(defectos_oed["origen"].value_counts().head(15))


origen
BUENAVENTURA    978
Name: count, dtype: int64


## Síntesis del Paso 10 (ampliada) — Refinamientos sobre Origen=Destino y Distancia=0

**974 de 978 (99,6%)** de los defectos `OPIM`/`OPTN` con Origen=Destino
tienen `distanciaTotal = 10` — la misma firma numérica de descontenerización
(`OHUB`/`OH3P`). Hipótesis más fuerte: no es un error de origen/destino,
es un **código de clase de documento mal asignado**. Solo 4 registros
(521, 521, 233.5, 505 km) son genuinamente atípicos y van a revisión
individual.

**14 de los 17 registros con `distanciaTotal = 0`** tienen `Origen ≠
Destino` con ciudades reales y distantes, y peso sustancial — son envíos
de largo recorrido a los que simplemente falta la distancia real. Es un
defecto limpio y acotado, distinto del fenómeno de descontenerización
(que casi no se superpone con distancia=0: solo 3 filas en común).

Ambos refinamientos documentados en el plan, §5.5.


## Paso 11 — Integridad referencial (§5.6)

Verificamos que los códigos de `idClaseDocumentoFwo` presentes estén todos
en el catálogo oficial (Anexo D), y que los 5 subconjuntos del archivo de
inconsistencias sean consistentes con nuestro dataset (Anexo B).

In [587]:
catalogo_clase_fwo = {"OH3P", "OHUB", "OHUC", "OPDS", "OPEX", "OPIM", "OPSC", "OPTI", "OPTN", "OPXL"}
codigos_presentes = set(df["idClaseDocumentoFwo"].unique())

print(f"Códigos presentes ({len(codigos_presentes)}): {sorted(codigos_presentes)}")
print(f"Huérfanos (en el dataset pero no en el catálogo): {codigos_presentes - catalogo_clase_fwo or 'ninguno'}")
print(f"Ausentes (en el catálogo pero no en el dataset): {sorted(catalogo_clase_fwo - codigos_presentes)}")


Códigos presentes (7): ['OH3P', 'OHUB', 'OPDS', 'OPEX', 'OPIM', 'OPTI', 'OPTN']
Huérfanos (en el dataset pero no en el catálogo): ninguno
Ausentes (en el catálogo pero no en el dataset): ['OHUC', 'OPSC', 'OPXL']


In [588]:
RAW_DIR = Path("../datos")
inconsistencias_path = RAW_DIR / "inconsistencias_viajes.xlsx"

# Las 4 hojas "limpias"
for hoja in ["Costo_0", "ImporteFO_0", "Peso_Neto_0", "ImporteFO_Negativo"]:
    hoja_df = pd.read_excel(inconsistencias_path, sheet_name=hoja)
    fos_hoja = set(hoja_df["FO"].dropna().astype("int64"))
    encontrados = fos_hoja & set(df["fo"])
    print(f"{hoja}: {len(fos_hoja)} FO, {len(encontrados)} encontrados en el dataset ({'OK' if len(encontrados) == len(fos_hoja) else 'FALTAN'})")

# Distancia_0: saltamos fila de diccionario (fila 2) y catálogo (filas 21-31)
raw = pd.read_excel(inconsistencias_path, sheet_name="Distancia_0", header=None)
columnas = raw.iloc[0]
datos_reales = raw.iloc[2:19].copy()
datos_reales.columns = columnas
fos_distancia0 = set(datos_reales["FO"].dropna().astype("int64"))
encontrados = fos_distancia0 & set(df["fo"])
print(f"Distancia_0: {len(fos_distancia0)} FO, {len(encontrados)} encontrados en el dataset ({'OK' if len(encontrados) == len(fos_distancia0) else 'FALTAN'})")


Costo_0: 869 FO, 869 encontrados en el dataset (OK)
ImporteFO_0: 183 FO, 183 encontrados en el dataset (OK)
Peso_Neto_0: 82 FO, 82 encontrados en el dataset (OK)
ImporteFO_Negativo: 1 FO, 1 encontrados en el dataset (OK)
Distancia_0: 17 FO, 17 encontrados en el dataset (OK)


## Paso 12 — Consistencia: tipo vs idMedioTransporteFo

`tipo` usa puntos y `idMedioTransporteFo` usa guiones para el mismo valor
— normalizamos el separador antes de comparar, para no contar falsas
discrepancias. También verificamos el valor basura ya conocido
(`idMedioTransporteFo = "1"`).


In [589]:
tipo_normalizado = df["tipo"].str.replace(".", "-", regex=False)
discrepancias = df[tipo_normalizado != df["idMedioTransporteFo"]]

print(f"Registros donde tipo (normalizado) != idMedioTransporteFo: {len(discrepancias)}")
print(discrepancias.groupby(["tipo", "idMedioTransporteFo"]).size().sort_values(ascending=False))

valor_basura = df[df["idMedioTransporteFo"] == "1"]
print(f"\nidMedioTransporteFo = '1' (valor basura): {len(valor_basura)}")


Registros donde tipo (normalizado) != idMedioTransporteFo: 5420
tipo        idMedioTransporteFo
C2.4.5T     TC_TURBO               1203
C2.8T       TC_SENCI                860
C2.2T       TC_CAMNETA              582
TC.3S2      TC-3S3                  423
TC.2S2      TC-2S3                  342
            FP-2S2                  271
C4.CM.25T   TC-4-MANOS              225
FP.2S3      FP-2S2                  203
TC.3S3      TC-3S2                  197
TC.2S2      FP-2S3                  195
TC.3S3      FP-3S3                  175
T.3S3.MM35  TC-3S3                  135
TC.2S3      TC-2S2                  130
            FP-2S2                   96
C2.4.5T     TC_SENCI                 81
TC.3.17T    TC-DOBTROQ               70
FP.2S2      TC-2S2                   68
T.2S2.MM18  TC-2S2                   36
FP.2S2      FP-2S3                   32
TC.2S3      FP-2S3                   31
TC.3S3      FP-3S2                   24
TC.2S2      TC-3S3                    9
C2.8T       TC_TURBO    

## Paso 13 — Nombres de ciudad similares

Comparamos todos los nombres de ciudad de origen/destino entre sí con
rapidfuzz, usando `token_set_ratio` (que maneja bien casos donde un nombre
es un subconjunto de palabras del otro, como "BOGOTA" dentro de
"BOGOTA, DISTRITO CAPITAL"). Umbral de similitud: 80.


In [590]:
from rapidfuzz import fuzz, process

ciudades = sorted(set(df["origen"].unique()) | set(df["destino"].unique()))
print(f"Total de nombres de ciudad únicos (origen ∪ destino): {len(ciudades)}")

candidatos = []
for i, ciudad in enumerate(ciudades):
    matches = process.extract(ciudad, ciudades[i+1:], scorer=fuzz.token_set_ratio, limit=3)
    for match_ciudad, score, _ in matches:
        if score >= 80:
            candidatos.append((ciudad, match_ciudad, round(score, 1)))

candidatos_df = pd.DataFrame(candidatos, columns=["ciudad_a", "ciudad_b", "similitud"]).sort_values("similitud", ascending=False)
candidatos_df


Total de nombres de ciudad únicos (origen ∪ destino): 146


,ciudad_a,ciudad_b,similitud
0,GIRARDOT,GIRARDOTA,94.1
2,SOACHA,SOCHA,90.9
1,SANTA ROSA DE CABAL,SANTA ROSA DE OSOS,83.9
3,TURBACO,TURBO,83.3


## Síntesis del Paso 12-13 — Consistencia

**`Tipo` vs `idMedioTransporteFo`:** 5.420 registros (13,1%) discrepan, no
184 como sugerían los 2 ejemplos citados en el plan — corregido en §5.4.
No cambia la recomendación (`Tipo` se descarta), la refuerza. Valor basura
`idMedioTransporteFo = "1"` confirmado: 2 registros.

**Nombres de ciudad:** comparación sistemática con rapidfuzz no encontró
ningún duplicado real — los 4 candidatos que salieron son municipios
colombianos distintos con nombres parecidos (falsos positivos). Los
ejemplos que el plan citaba (variantes de Bogotá/Buga) no existen en este
dataset — se corrige en el plan, §5.4.

Con esto, las 6 dimensiones de calidad evaluables con este dato quedan
cubiertas: completitud, unicidad (§4), validez, consistencia, exactitud e
integridad referencial. Falta solo oportunidad/cobertura temporal (§5.7).


## Paso 14 — Cobertura temporal

Registros por mes, para confirmar la estabilidad mensual y el mes parcial
(abril 2026) ya documentados en el plan. También revisamos si hay días
completos sin ningún registro dentro de la ventana.


In [591]:
por_mes = df["fechaCreacionFo"].dt.to_period("M").value_counts().sort_index()
print(por_mes)

dias_con_datos = set(df["fechaCreacionFo"].dt.date)
rango_completo = pd.date_range(df["fechaCreacionFo"].min(), df["fechaCreacionFo"].max(), freq="D").date
dias_sin_datos = sorted(set(rango_completo) - dias_con_datos)
print(f"\nDías dentro de la ventana sin ningún registro: {len(dias_sin_datos)}")
print(dias_sin_datos[:15])


fechaCreacionFo
2025-04    2959
2025-05    3402
2025-06    3245
2025-07    3821
2025-08    3306
2025-09    3582
2025-10    3380
2025-11    3144
2025-12    2961
2026-01    3396
2026-02    3798
2026-03    3720
2026-04     523
Freq: M, Name: count, dtype: int64

Días dentro de la ventana sin ningún registro: 1
[datetime.date(2025, 12, 25)]


## Síntesis del notebook 02 — Calidad del dato

**Cobertura temporal:** confirmada estable (2.959-3.821 registros/mes),
abril 2026 parcial (523, corte al día 7). Único día sin registros en toda
la ventana: 25-dic-2025 (Navidad) — esperado, no es un defecto.

### Tabla resumen de calidad (§5.8)

| Dimensión | Hallazgo | Registros | % | Severidad | Decisión | Responsable |
|---|---|---|---|---|---|---|
| Completitud | Nulos estructurales equipo (PKG+PRD) | 7.496 | 18,2% | N/A — por diseño | Ninguna | — |
| Completitud | TUR con `claseEquipoFop` nula | 3 | 0,01% | Baja | Marcar + consultar | Negocio |
| Unicidad | `FO` repetido — carga consolidada (hipótesis fuerte) | 128 grupos / 258 filas | 0,6% | N/A — no es defecto | Marcar bandera "consolidado" + confirmar | Negocio |
| Validez | Flete negativo | 1 | 0,00% | Media | Consultar | Negocio |
| Validez | Vehículo no asignado (`GENERICO_3S3`) | 1 | 0,00% | Baja | Marcar | Negocio |
| Validez | Placas venezolanas (`OPTI` a Venezuela) | 2 | 0,00% | N/A — no es defecto | Ninguna | — |
| Exactitud | Distancia=0 con Origen≠Destino (dato faltante real) | 14 | 0,03% | Media | Marcar + consultar | Negocio |
| Exactitud | Peso = 0 | 82 | 0,20% | Media | Consultar | Negocio |
| Exactitud | Tarifa = 0 | 874 | 2,12% | Alta | Marcar + consultar | Negocio |
| Exactitud | Flete = 0 | 183 | 0,44% | Media | Consultar | Negocio |
| Exactitud | Margen negativo | 3.839 | 9,31% | Alta | Consultar | Negocio |
| Exactitud | `OPIM`/`OPTN` con Origen=Destino, probable código mal asignado | 974 | 2,36% | Media | Marcar + consultar recodificación a `OHUB`/`OH3P` | Negocio |
| Exactitud | `OPIM`/`OPTN` con Origen=Destino genuinamente atípico | 4 | 0,01% | Baja | Revisión individual | Equipo de datos |
| Consistencia | `Tipo` ≠ `idMedioTransporteFo` | 5.420 | 13,1% | N/A — `Tipo` ya descartado | Excluir `Tipo` del análisis | — |
| Consistencia | `idMedioTransporteFo = "1"` (valor basura) | 2 | 0,00% | Baja | Marcar | Negocio |
| Integridad ref. | Líneas de negocio del catálogo ausentes (`OHUC`,`OPSC`,`OPXL`) | 3 de 10 | — | Media | Consultar alcance del set | Negocio |
| Oportunidad | Mes parcial (abril 2026) | 523 | 1,27% | N/A — esperado | Excluir de comparaciones mensuales completas | — |
